# Task: Denoising diffusion model for CIFAR10

– Load CIFAR10 with `tf.keras.datasets.cifar10.load_data()`  
– Keep a single class: label = last digit of student ID = **9** (trucks)  
– Train a denoising diffusion model with suitable hyperparameters  
– Generate images of the chosen class  
– Visualize generations for different numbers of reverse diffusion steps  
– Interpolation between two training images (image chain)

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import matplotlib.pyplot as plt
import math

import keras
from keras import layers, models, optimizers, callbacks, metrics, losses, activations
import tensorflow as tf

In [ ]:
# Last digit of student ID = 9 -> class "trucks" (CIFAR10: 0=airplane,1=car,...,9=truck)
STUDENT_LABEL = 9

IMAGE_SIZE = 32   # CIFAR10 est 32x32
BATCH_SIZE = 64
NOISE_EMBEDDING_SIZE = 32
EMA = 0.999
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 30

## 1. Load CIFAR10 and filter for class 9 (trucks)

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Keep only the class = last digit of student ID (9 = trucks)
mask_train = (y_train.flatten() == STUDENT_LABEL)
mask_test = (y_test.flatten() == STUDENT_LABEL)
x_train = x_train[mask_train].astype(np.float32) / 255.0
y_train = y_train[mask_train]
x_test = x_test[mask_test].astype(np.float32) / 255.0

print(f"Train images (class {STUDENT_LABEL}): {len(x_train)}")
print(f"Test images: {len(x_test)}")

In [ ]:
def preprocess(img):
    return tf.cast(img, tf.float32)

train_ds = tf.data.Dataset.from_tensor_slices((x_train,))
train_ds = train_ds.map(lambda x: (preprocess(x),))
train_ds = train_ds.shuffle(buffer_size=len(x_train)).batch(BATCH_SIZE, drop_remainder=True)

## 2. Diffusion schedule (offset cosine)

In [ ]:
def offset_cosine_diffusion_schedule(diffusion_times):
    diffusion_times = tf.cast(diffusion_times, tf.float32)
    min_signal_rate = 0.02
    max_signal_rate = 0.95
    start_angle = tf.acos(tf.constant(max_signal_rate, dtype=tf.float32))
    end_angle = tf.acos(tf.constant(min_signal_rate, dtype=tf.float32))
    diffusion_angles = start_angle + diffusion_times * (end_angle - start_angle)
    signal_rates = tf.cos(diffusion_angles)
    noise_rates = tf.sin(diffusion_angles)
    return noise_rates, signal_rates

## 3. U-Net : sinusoidal embedding, ResidualBlock, DownBlock, UpBlock

In [ ]:
@keras.saving.register_keras_serializable()
def sinusoidal_embedding(x):
    frequencies = tf.exp(
        tf.linspace(
            tf.math.log(1.0),
            tf.math.log(1000.0),
            NOISE_EMBEDDING_SIZE // 2,
        )
    )
    angular_speeds = 2.0 * math.pi * frequencies
    embeddings = tf.concat([tf.sin(angular_speeds*x), tf.cos(angular_speeds*x)], axis=3)
    return embeddings

def ResidualBlock(width):
    def apply(x):
        input_width = x.shape[3]
        if input_width == width:
            residual = x
        else:
            residual = layers.Conv2D(width, kernel_size=1)(x)
        x = layers.BatchNormalization(center=False, scale=False)(x)
        x = layers.Conv2D(width, kernel_size=3, padding="same", activation=activations.swish)(x)
        x = layers.Conv2D(width, kernel_size=3, padding="same")(x)
        x = layers.Add()([x, residual])
        return x
    return apply

def DownBlock(width, block_depth):
    def apply(x):
        x, skips = x
        for _ in range(block_depth):
            x = ResidualBlock(width)(x)
            skips.append(x)
        x = layers.AveragePooling2D(pool_size=2)(x)
        return x
    return apply

def UpBlock(width, block_depth):
    def apply(x):
        x, skips = x
        x = layers.UpSampling2D(size=2, interpolation="bilinear")(x)
        for _ in range(block_depth):
            x = layers.Concatenate()([x, skips.pop()])
            x = ResidualBlock(width)(x)
        return x
    return apply

In [ ]:
def build_unet(name="unet"):
    """Build a U-Net model (call twice to get network and EMA network without clone_model)."""
    noisy_images = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    x = layers.Conv2D(32, kernel_size=1)(noisy_images)
    noise_variances = layers.Input(shape=(1, 1, 1))
    noise_embedding = layers.Lambda(sinusoidal_embedding)(noise_variances)
    noise_embedding = layers.UpSampling2D(size=IMAGE_SIZE, interpolation="nearest")(noise_embedding)
    x = layers.Concatenate()([x, noise_embedding])
    skips = []
    x = DownBlock(32, block_depth=2)([x, skips])
    x = DownBlock(64, block_depth=2)([x, skips])
    x = DownBlock(96, block_depth=2)([x, skips])
    x = ResidualBlock(128)(x)
    x = ResidualBlock(128)(x)
    x = UpBlock(96, block_depth=2)([x, skips])
    x = UpBlock(64, block_depth=2)([x, skips])
    x = UpBlock(32, block_depth=2)([x, skips])
    x = layers.Conv2D(3, kernel_size=1, kernel_initializer="zeros")(x)
    return models.Model([noisy_images, noise_variances], x, name=name)

unet = build_unet(name="unet")

## 4. Diffusion model (denoising)

In [ ]:
class DiffusionModel(models.Model):
    def __init__(self):
        super().__init__()
        self.normalizer = layers.Normalization()
        self.network = unet
        self.ema_network = build_unet(name="unet_ema")
        self.diffusion_schedule = offset_cosine_diffusion_schedule

    def compile(self, **kwargs):
        super().compile(**kwargs)
        self.noise_loss_tracker = metrics.Mean(name="n_loss")

    @property
    def metrics(self):
        return [self.noise_loss_tracker]

    def denormalize(self, images):
        images = self.normalizer.mean + images * self.normalizer.variance**0.5
        return tf.clip_by_value(images, 0.0, 1.0)

    def denoise(self, noisy_images, noise_rates, signal_rates, training):
        network = self.network if training else self.ema_network
        pred_noises = network([noisy_images, noise_rates**2], training=training)
        pred_images = (noisy_images - noise_rates * pred_noises) / signal_rates
        return pred_noises, pred_images

    def reverse_diffusion(self, initial_noise, diffusion_steps):
        num_images = initial_noise.shape[0]
        step_size = 1.0 / diffusion_steps
        current_images = initial_noise
        for step in range(diffusion_steps):
            diffusion_times = tf.ones((num_images, 1, 1, 1)) - step * step_size
            noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
            pred_noises, pred_images = self.denoise(current_images, noise_rates, signal_rates, training=False)
            next_diffusion_times = diffusion_times - step_size
            next_noise_rates, next_signal_rates = self.diffusion_schedule(next_diffusion_times)
            current_images = next_signal_rates * pred_images + next_noise_rates * pred_noises
        return pred_images

    def reverse_diffusion_from_time(self, initial_noise, start_time, diffusion_steps):
        """Reverse diffusion en partant d'un état à start_time (0<=start_time<=1)."""
        num_images = initial_noise.shape[0]
        step_size = 1.0 / diffusion_steps
        steps_to_do = int(round(start_time / step_size))
        if steps_to_do <= 0:
            return self.denormalize(initial_noise)
        current_images = initial_noise
        t = start_time
        for _ in range(steps_to_do):
            diffusion_times = tf.ones((num_images, 1, 1, 1)) * t
            noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
            pred_noises, pred_images = self.denoise(current_images, noise_rates, signal_rates, training=False)
            t = t - step_size
            if t < 0:
                t = 0.0
            next_diffusion_times = tf.ones((num_images, 1, 1, 1)) * t
            next_noise_rates, next_signal_rates = self.diffusion_schedule(next_diffusion_times)
            current_images = next_signal_rates * pred_images + next_noise_rates * pred_noises
        return pred_images

    def generate(self, num_images, diffusion_steps, initial_noise=None):
        if initial_noise is None:
            initial_noise = tf.random.normal(shape=(num_images, IMAGE_SIZE, IMAGE_SIZE, 3))
        generated = self.reverse_diffusion(initial_noise, diffusion_steps)
        return self.denormalize(generated)

    def train_step(self, data):
        images = data[0] if isinstance(data, (list, tuple)) else data
        images = self.normalizer(images, training=True)
        noises = tf.random.normal(shape=(BATCH_SIZE, IMAGE_SIZE, IMAGE_SIZE, 3))
        diffusion_times = tf.random.uniform(shape=(BATCH_SIZE, 1, 1, 1), minval=0.0, maxval=1.0)
        noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
        noisy_images = signal_rates * images + noise_rates * noises
        with tf.GradientTape() as tape:
            pred_noises, _ = self.denoise(noisy_images, noise_rates, signal_rates, training=True)
            noise_loss = self.loss(noises, pred_noises)
        gradients = tape.gradient(noise_loss, self.network.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.network.trainable_weights))
        self.noise_loss_tracker.update_state(noise_loss)
        for w, ema_w in zip(self.network.weights, self.ema_network.weights):
            ema_w.assign(EMA * ema_w + (1 - EMA) * w)
        return {m.name: m.result() for m in self.metrics}

In [ ]:
ddm = DiffusionModel()
ddm.normalizer.adapt(train_ds.map(lambda x: x[0]))
# Initialize EMA network with same weights as main network
for w, ema_w in zip(ddm.network.weights, ddm.ema_network.weights):
    ema_w.assign(w)
ddm.compile(optimizer=optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
            loss=losses.MeanSquaredError())

## 5. Training

In [ ]:
history = ddm.fit(train_ds, epochs=EPOCHS)

## 6. Generation and visualization for different numbers of steps

In [ ]:
def display(images, n=None, title=None):
    if n is None:
        n = len(images)
    images = np.array(images)[:n]
    if images.ndim == 4 and images.shape[-1] == 3:
        pass
    else:
        return
    rows = (n + 7) // 8
    cols = min(n, 8)
    fig, ax = plt.subplots(rows, cols, figsize=(cols*1.5, rows*1.5))
    if rows == 1 and cols == 1:
        ax = np.array([[ax]])
    elif rows == 1:
        ax = ax.reshape(1, -1)
    elif cols == 1:
        ax = ax.reshape(-1, 1)
    for i in range(rows * cols):
        r, c = i // cols, i % cols
        if i < n:
            ax[r, c].imshow(np.clip(images[i], 0, 1))
        ax[r, c].axis("off")
    if title:
        fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
steps_list = [5, 10, 20, 50]
np.random.seed(42)
tf.random.set_seed(42)

for num_steps in steps_list:
    gen = ddm.generate(num_images=8, diffusion_steps=num_steps)
    display(gen.numpy(), n=8, title=f"Generation — {num_steps} reverse diffusion steps")

## 7. Image chain: interpolation between two training images

In [ ]:
def spherical_interpolation(a, b, t):
    return np.sin(t * math.pi / 2) * a + np.cos(t * math.pi / 2) * b

# Two images from the training set (normalized as during training)
idx_a, idx_b = 0, 100
img_a = tf.constant(x_train[idx_a:idx_a+1])   # (1,32,32,3)
img_b = tf.constant(x_train[idx_b:idx_b+1])

ddm.normalizer.adapt(train_ds.map(lambda x: x[0]))
norm_a = ddm.normalizer(img_a)
norm_b = ddm.normalizer(img_b)

# Forward diffusion at the same time t for both images (different noises)
t_mid = 0.5
np.random.seed(123)
noise_a = np.random.randn(1, IMAGE_SIZE, IMAGE_SIZE, 3).astype(np.float32)
noise_b = np.random.randn(1, IMAGE_SIZE, IMAGE_SIZE, 3).astype(np.float32)
noise_rates, signal_rates = offset_cosine_diffusion_schedule(tf.constant([[t_mid]]))
noisy_a = (signal_rates * norm_a + noise_rates * noise_a).numpy()
noisy_b = (signal_rates * norm_b + noise_rates * noise_b).numpy()

# Interpolation in the noisy space
n_interp = 9
interp_noises = [spherical_interpolation(noisy_a[0], noisy_b[0], t) for t in np.linspace(0, 1, n_interp)]
interp_noises = tf.convert_to_tensor(np.stack(interp_noises))

# Reverse diffusion from these states (from t_mid to 0)
diffusion_steps = 50
start_step = int(t_mid * diffusion_steps)
step_size = 1.0 / diffusion_steps
current = tf.cast(interp_noises, tf.float32)
for step in range(start_step):
    t = t_mid - (step + 1) * step_size
    t = max(0.0, t)
    diffusion_times = tf.ones((n_interp, 1, 1, 1), dtype=tf.float32) * tf.constant(t + step_size, dtype=tf.float32)
    noise_rates, signal_rates = ddm.diffusion_schedule(diffusion_times)
    pred_noises, pred_images = ddm.denoise(current, noise_rates, signal_rates, training=False)
    next_noise_rates, next_signal_rates = ddm.diffusion_schedule(tf.ones((n_interp, 1, 1, 1), dtype=tf.float32) * tf.constant(t, dtype=tf.float32))
    current = next_signal_rates * pred_images + next_noise_rates * pred_noises

chain = ddm.denormalize(current).numpy()
display(chain, n=n_interp, title="Interpolation chain between two training images (trucks)")

In [ ]:
# Also display the two original images for comparison
display(np.stack([x_train[idx_a], x_train[idx_b]]), n=2, title="Two training images (origin of the interpolation)")